# NEH On NEH

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_onlyval_5e4_GradAug_Speckle_RC_va/epoch_27.pth"


CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_reduced_split_val_20.csv"


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 2. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET (PIL, same as training/testing)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

# =====================================================
# 4. TRANSFORM (IDENTICAL TO TRAIN/VAL TEST)
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

dataset = UCSDDataset(CSV_PATH)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pil_collate_fn,
)

# =====================================================
# 5. GRADAUG MODEL DEFINITION (EXACT)
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def _forward_stem(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x

    def _head(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def forward_full(self, x):
        x = self._forward_stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self._head(x)

# =====================================================
# 6. LOAD CHECKPOINT
# =====================================================
print(f"\n--- Loading GradAug model ---")
print(CHECKPOINT_PATH)

model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False)
model = model.to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)

model.eval()

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        labels = labels.numpy()

        imgs = torch.stack(
            [test_transform(img) for img in imgs_pil]
        ).to(DEVICE)

        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels)

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 8. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )


specificity = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"GradAug EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.6f}")
print(f"{'AUC':<30} | {auc:.6f}")

print(f"{'F1':<30} | {f1:.6f}")
print(f"{'F2 (β=2)':<30} | {f2:.6f}")
print(f"{'Macro F1':<30} | {macro_f1:.6f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.6f}")


print(f"{'Precision':<30} | {precision:.6f}")
print(f"{'Recall':<30} | {recall:.6f}")

print(f"{'Macro Precision':<30} | {macro_precision:.6f}")
print(f"{'Macro Recall':<30} | {macro_recall:.6f}")

print(f"{'Specificity':<30} | {specificity:.6f}")
print(f"{'NPV':<30} | {npv:.6f}")

print(f"{'Cohen Kappa':<30} | {kappa:.6f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.6f}")
print("=" * 60)

# ======================================================================
# GRADAUG MODEL — UCSD TEST EVALUATION
# ======================================================================
# Metric                              | Value
# ----------------------------------------------------------------------
# Accuracy                            | 0.9211
# ROC AUC                             | 0.9730
# F1                                  | 0.9204
# Macro F1                            | 0.9210
# Macro Precision                     | 0.9214
# Macro Recall                        | 0.9217
# Specificity                         | 0.9009
# NPV                                 | 0.9434
# Cohen Kappa                         | 0.8422
# ----------------------------------------------------------------------
# Avg Set Size (τ=0.9)                | 1.093
# Singleton Rate % (τ=0.9)            | 90.71
# ECE                                 | 0.0572
# ======================================================================


--- Loading GradAug model ---
/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_onlyval_5e4_GradAug_Speckle_RC_va/epoch_27.pth


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


KeyboardInterrupt: 

In [5]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. PATHS & CONFIG
# =====================================================
# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth"
# VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/venn_abers_fitted.pkl"

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 3. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. DATASET & TRANSFORM
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=pil_collate_fn)

# =====================================================
# 5. GRADAUG MODEL DEFINITION
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)
        self.conv1, self.bn1, self.relu, self.maxpool = base.conv1, base.bn1, base.relu, base.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = base.layer1, base.layer2, base.layer3, base.layer4
        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def forward_full(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# =====================================================
# 6. LOAD CHECKPOINT & VA
# =====================================================
print(f"\n--- Loading GradAug Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()

print(f"--- Loading Venn-Abers: {os.path.basename(VA_PICKLE_PATH)} ---")
with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []
with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        imgs = torch.stack([test_transform(img) for img in imgs_pil]).to(DEVICE)
        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 8. RESULTS DISPLAY
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"GradAug EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)


--- Loading GradAug Model: epoch_11.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference: 100%|██████████| 21/21 [00:04<00:00,  4.64it/s]



Applying Venn-Abers Calibration...

GradAug EVALUATION: epoch_11.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.9226          | 0.9040         
AUC                            | 0.9840          | 0.9842         
F1                             | 0.9249          | 0.9088         
F2 (β=2)                       | 0.9217          | 0.9023         
Macro F1                       | 0.9225          | 0.9038         
Macro F2 (β=2)                 | 0.9228          | 0.9039         
Precision                      | 0.8725          | 0.8420         
Recall                         | 0.9840          | 0.9872         
Macro Precision                | 0.9277          | 0.9138         
Macro Recall                   | 0.9244          | 0.9065         
Specificity                    | 0.8649          | 0.8258         
NPV                            | 0.9829          |

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth"


CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv"


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 2. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):

    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET (PIL, same as training/testing)
# =====================================================

class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

# =====================================================
# 4. TRANSFORM (IDENTICAL TO TRAIN/VAL TEST)
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

dataset = UCSDDataset(CSV_PATH)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pil_collate_fn,
)

# =====================================================
# 5. GRADAUG MODEL DEFINITION (EXACT)
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def _forward_stem(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x

    def _head(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def forward_full(self, x):
        x = self._forward_stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self._head(x)

# =====================================================
# 6. LOAD CHECKPOINT
# =====================================================
print(f"\n--- Loading GradAug model ---")
print(CHECKPOINT_PATH)

model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False)
model = model.to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)

model.eval()

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        labels = labels.numpy()

        imgs = torch.stack(
            [test_transform(img) for img in imgs_pil]
        ).to(DEVICE)

        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels)

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 8. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )



specificity = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"GradAug EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)




--- Loading GradAug model ---
/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 21/21 [00:05<00:00,  3.87it/s]


GradAug EVALUATION — epoch_11.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.9226
AUC                            | 0.9840
F1                             | 0.9249
F2 (β=2)                       | 0.9595
Macro F1                       | 0.9225
Macro F2 (β=2)                 | 0.9251
Precision                      | 0.8725
Recall                         | 0.9840
Macro Precision                | 0.9277
Macro Recall                   | 0.9244
Specificity                    | 0.8649
NPV                            | 0.9829
Cohen Kappa                    | 0.8456
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.119
Singleton (%) (τ=0.9)          | 88.08
ECE                            | 0.0416


# NEH on UCSD

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth"


CSV_PATH = (
    "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/"
    "UCSD_2/ucsd_filtered.csv"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 2. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET (PIL, same as training/testing)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

# =====================================================
# 4. TRANSFORM (IDENTICAL TO TRAIN/VAL TEST)
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

dataset = UCSDDataset(CSV_PATH)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pil_collate_fn,
)

# =====================================================
# 5. GRADAUG MODEL DEFINITION (EXACT)
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def _forward_stem(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x

    def _head(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def forward_full(self, x):
        x = self._forward_stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self._head(x)

# =====================================================
# 6. LOAD CHECKPOINT
# =====================================================
print(f"\n--- Loading GradAug model ---")
print(CHECKPOINT_PATH)

model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False)
model = model.to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)

model.eval()

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        labels = labels.numpy()

        imgs = torch.stack(
            [test_transform(img) for img in imgs_pil]
        ).to(DEVICE)

        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels)

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 8. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )
specificity = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"GradAug EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading GradAug model ---
/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 3014/3014 [12:34<00:00,  4.00it/s]


GradAug EVALUATION — epoch_11.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.8501
AUC                            | 0.9527
F1                             | 0.8719
F2 (β=2)                       | 0.9269
Macro F1                       | 0.8456
Macro F2 (β=2)                 | 0.8491
Precision                      | 0.7933
Recall                         | 0.9677
Macro Precision                | 0.8728
Macro Recall                   | 0.8434
Specificity                    | 0.7191
NPV                            | 0.9523
Cohen Kappa                    | 0.6955
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.223
Singleton (%) (τ=0.9)          | 77.73
ECE                            | 0.0793


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth"


CSV_PATH = (
    "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/"
    "UCSD_2/ucsd_filtered.csv"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 2. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET (PIL, same as training/testing)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

# =====================================================
# 4. TRANSFORM (IDENTICAL TO TRAIN/VAL TEST)
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

dataset = UCSDDataset(CSV_PATH)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pil_collate_fn,
)

# =====================================================
# 5. GRADAUG MODEL DEFINITION (EXACT)
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def _forward_stem(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x

    def _head(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def forward_full(self, x):
        x = self._forward_stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self._head(x)

# =====================================================
# 6. LOAD CHECKPOINT
# =====================================================
print(f"\n--- Loading GradAug model ---")
print(CHECKPOINT_PATH)

model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False)
model = model.to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)

model.eval()

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        labels = labels.numpy()

        imgs = torch.stack(
            [test_transform(img) for img in imgs_pil]
        ).to(DEVICE)

        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels)

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 8. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )
specificity = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"GradAug EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading GradAug model ---
/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 3014/3014 [12:26<00:00,  4.04it/s]


GradAug EVALUATION — epoch_9.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.7298
AUC                            | 0.9165
F1                             | 0.7929
F2 (β=2)                       | 0.8965
Macro F1                       | 0.7020
Macro F2 (β=2)                 | 0.7326
Precision                      | 0.6650
Recall                         | 0.9819
Macro Precision                | 0.8110
Macro Recall                   | 0.7153
Specificity                    | 0.4488
NPV                            | 0.9570
Cohen Kappa                    | 0.4428
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.267
Singleton (%) (τ=0.9)          | 73.32
ECE                            | 0.1869


In [6]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. PATHS & CONFIG
# =====================================================

# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth"
# VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/venn_abers_fitted.pkl"


CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 3. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. DATASET & TRANSFORM
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=pil_collate_fn)

# =====================================================
# 5. GRADAUG MODEL DEFINITION
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)
        self.conv1, self.bn1, self.relu, self.maxpool = base.conv1, base.bn1, base.relu, base.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = base.layer1, base.layer2, base.layer3, base.layer4
        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def forward_full(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# =====================================================
# 6. LOAD CHECKPOINT & VA
# =====================================================
print(f"\n--- Loading GradAug Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()

print(f"--- Loading Venn-Abers: {os.path.basename(VA_PICKLE_PATH)} ---")
with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []
with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        imgs = torch.stack([test_transform(img) for img in imgs_pil]).to(DEVICE)
        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 8. RESULTS DISPLAY
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"GradAug EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)


--- Loading GradAug Model: epoch_11.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference: 100%|██████████| 3014/3014 [10:47<00:00,  4.66it/s]



Applying Venn-Abers Calibration...

GradAug EVALUATION: epoch_11.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.8501          | 0.8252         
AUC                            | 0.9527          | 0.9514         
F1                             | 0.8719          | 0.8548         
F2 (β=2)                       | 0.8461          | 0.8186         
Macro F1                       | 0.8456          | 0.8177         
Macro F2 (β=2)                 | 0.8415          | 0.8126         
Precision                      | 0.7933          | 0.7603         
Recall                         | 0.9677          | 0.9762         
Macro Precision                | 0.8728          | 0.8608         
Macro Recall                   | 0.8434          | 0.8166         
Specificity                    | 0.7191          | 0.6570         
NPV                            | 0.9523          |

# NEH on DHU

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth"


CSV_PATH = (
    "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/"
    "UCSD_2/dhu_filtered.csv"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 2. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET (PIL, same as training/testing)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

# =====================================================
# 4. TRANSFORM (IDENTICAL TO TRAIN/VAL TEST)
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

dataset = UCSDDataset(CSV_PATH)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pil_collate_fn,
)

# =====================================================
# 5. GRADAUG MODEL DEFINITION (EXACT)
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def _forward_stem(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x

    def _head(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def forward_full(self, x):
        x = self._forward_stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self._head(x)

# =====================================================
# 6. LOAD CHECKPOINT
# =====================================================
print(f"\n--- Loading GradAug model ---")
print(CHECKPOINT_PATH)

model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False)
model = model.to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)

model.eval()

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        labels = labels.numpy()

        imgs = torch.stack(
            [test_transform(img) for img in imgs_pil]
        ).to(DEVICE)

        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels)

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 8. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )
specificity = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"GradAug EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading GradAug model ---
/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 90/90 [00:22<00:00,  4.04it/s]


GradAug EVALUATION — epoch_11.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.8862
AUC                            | 0.9818
F1                             | 0.8977
F2 (β=2)                       | 0.9462
Macro F1                       | 0.8847
Macro F2 (β=2)                 | 0.8879
Precision                      | 0.8272
Recall                         | 0.9815
Macro Precision                | 0.9017
Macro Recall                   | 0.8845
Specificity                    | 0.7875
NPV                            | 0.9762
Cohen Kappa                    | 0.7716
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.228
Singleton (%) (τ=0.9)          | 77.17
ECE                            | 0.0482


In [7]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. PATHS & CONFIG
# =====================================================

# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth"
# VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/venn_abers_fitted.pkl"


CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 3. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. DATASET & TRANSFORM
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=pil_collate_fn)

# =====================================================
# 5. GRADAUG MODEL DEFINITION
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)
        self.conv1, self.bn1, self.relu, self.maxpool = base.conv1, base.bn1, base.relu, base.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = base.layer1, base.layer2, base.layer3, base.layer4
        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def forward_full(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# =====================================================
# 6. LOAD CHECKPOINT & VA
# =====================================================
print(f"\n--- Loading GradAug Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()

print(f"--- Loading Venn-Abers: {os.path.basename(VA_PICKLE_PATH)} ---")
with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []
with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        imgs = torch.stack([test_transform(img) for img in imgs_pil]).to(DEVICE)
        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 8. RESULTS DISPLAY
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"GradAug EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)


--- Loading GradAug Model: epoch_11.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference: 100%|██████████| 90/90 [00:24<00:00,  3.75it/s]



Applying Venn-Abers Calibration...

GradAug EVALUATION: epoch_11.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.8862          | 0.8503         
AUC                            | 0.9818          | 0.9817         
F1                             | 0.8977          | 0.8700         
F2 (β=2)                       | 0.8838          | 0.8454         
Macro F1                       | 0.8847          | 0.8467         
Macro F2 (β=2)                 | 0.8827          | 0.8438         
Precision                      | 0.8272          | 0.7795         
Recall                         | 0.9815          | 0.9842         
Macro Precision                | 0.9017          | 0.8785         
Macro Recall                   | 0.8845          | 0.8478         
Specificity                    | 0.7875          | 0.7114         
NPV                            | 0.9762          |

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth"


CSV_PATH = (
    "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/"
    "UCSD_2/dhu_filtered.csv"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 2. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET (PIL, same as training/testing)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

# =====================================================
# 4. TRANSFORM (IDENTICAL TO TRAIN/VAL TEST)
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

dataset = UCSDDataset(CSV_PATH)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pil_collate_fn,
)

# =====================================================
# 5. GRADAUG MODEL DEFINITION (EXACT)
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def _forward_stem(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x

    def _head(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def forward_full(self, x):
        x = self._forward_stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self._head(x)

# =====================================================
# 6. LOAD CHECKPOINT
# =====================================================
print(f"\n--- Loading GradAug model ---")
print(CHECKPOINT_PATH)

model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False)
model = model.to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)

model.eval()

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        labels = labels.numpy()

        imgs = torch.stack(
            [test_transform(img) for img in imgs_pil]
        ).to(DEVICE)

        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels)

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 8. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

specificity = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"GradAug EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



--- Loading GradAug model ---
/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth


Inference: 100%|██████████| 90/90 [00:22<00:00,  4.06it/s]


GradAug EVALUATION — epoch_9.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.7602
AUC                            | 0.9765
F1                             | 0.8073
F2 (β=2)                       | 0.9063
Macro F1                       | 0.7450
Macro F2 (β=2)                 | 0.7696
Precision                      | 0.6830
Recall                         | 0.9870
Macro Precision                | 0.8289
Macro Recall                   | 0.7561
Specificity                    | 0.5252
NPV                            | 0.9749
Cohen Kappa                    | 0.5164
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.319
Singleton (%) (τ=0.9)          | 68.10
ECE                            | 0.1417


# NEH on OCT C8

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth"


CSV_PATH = (
    "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/"
    "UCSD_2/octC8_filtered.csv"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 2. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET (PIL, same as training/testing)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

# =====================================================
# 4. TRANSFORM (IDENTICAL TO TRAIN/VAL TEST)
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

dataset = UCSDDataset(CSV_PATH)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pil_collate_fn,
)

# =====================================================
# 5. GRADAUG MODEL DEFINITION (EXACT)
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def _forward_stem(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x

    def _head(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def forward_full(self, x):
        x = self._forward_stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self._head(x)

# =====================================================
# 6. LOAD CHECKPOINT
# =====================================================
print(f"\n--- Loading GradAug model ---")
print(CHECKPOINT_PATH)

model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False)
model = model.to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)

model.eval()

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        labels = labels.numpy()

        imgs = torch.stack(
            [test_transform(img) for img in imgs_pil]
        ).to(DEVICE)

        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels)

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 8. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

specificity = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"GradAug EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading GradAug model ---
/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 180/180 [00:43<00:00,  4.15it/s]


GradAug EVALUATION — epoch_11.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.8172
AUC                            | 0.8907
F1                             | 0.8383
F2 (β=2)                       | 0.8767
Macro F1                       | 0.8141
Macro F2 (β=2)                 | 0.8157
Precision                      | 0.7812
Recall                         | 0.9043
Macro Precision                | 0.8270
Macro Recall                   | 0.8129
Specificity                    | 0.7215
NPV                            | 0.8728
Cohen Kappa                    | 0.6308
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.315
Singleton (%) (τ=0.9)          | 68.53
ECE                            | 0.0884


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/venn_abers_fitted.pkl"


# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/epoch_11.pth"
# VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified_with_speckle/venn_abers_fitted.pkl"

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered_imbalanced.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 3. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. DATASET & TRANSFORM
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=pil_collate_fn)

# =====================================================
# 5. GRADAUG MODEL DEFINITION
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)
        self.conv1, self.bn1, self.relu, self.maxpool = base.conv1, base.bn1, base.relu, base.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = base.layer1, base.layer2, base.layer3, base.layer4
        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def forward_full(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# =====================================================
# 6. LOAD CHECKPOINT & VA
# =====================================================
print(f"\n--- Loading GradAug Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()

print(f"--- Loading Venn-Abers: {os.path.basename(VA_PICKLE_PATH)} ---")
with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []
with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        imgs = torch.stack([test_transform(img) for img in imgs_pil]).to(DEVICE)
        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 8. RESULTS DISPLAY
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"GradAug EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)

# Applying Venn-Abers Calibration... for resize and speckle noise

# ===========================================================================
# GradAug EVALUATION: epoch_11.pth
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# Accuracy                       | 0.8504          | 0.8498         
# AUC                            | 0.8964          | 0.8936         
# F1                             | 0.8926          | 0.8950         
# F2 (β=2)                       | 0.8495          | 0.8469         
# Macro F1                       | 0.8231          | 0.8158         
# Macro F2 (β=2)                 | 0.8185          | 0.8050         
# Precision                      | 0.8757          | 0.8569         
# Recall                         | 0.9102          | 0.9366         
# Macro Precision                | 0.8321          | 0.8430         
# Macro Recall                   | 0.8159          | 0.7997         
# Specificity                    | 0.7215          | 0.6629         
# NPV                            | 0.7885          | 0.8291         
# Cohen Kappa                    | 0.6465          | 0.6335         
# Avg Set Size                   | 1.2826          | 1.7222         
# Singleton %                    | 71.7422         | 27.7816        
# ECE                            | 0.0642          | 0.0928         
# ===========================================================================





--- Loading GradAug Model: epoch_9.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference: 100%|██████████| 270/270 [00:58<00:00,  4.60it/s]



Applying Venn-Abers Calibration...

GradAug EVALUATION: epoch_9.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.8079          | 0.7603         
AUC                            | 0.8296          | 0.8284         
F1                             | 0.8725          | 0.8490         
F2 (β=2)                       | 0.7966          | 0.7333         
Macro F1                       | 0.7416          | 0.6341         
Macro F2 (β=2)                 | 0.7230          | 0.6218         
Precision                      | 0.7981          | 0.7452         
Recall                         | 0.9621          | 0.9864         
Macro Precision                | 0.8258          | 0.8241         
Macro Recall                   | 0.7189          | 0.6297         
Specificity                    | 0.4756          | 0.2730         
NPV                            | 0.8534          | 

: 

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth"


CSV_PATH = (
    "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/"
    "UCSD_2/octC8_filtered.csv"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 2. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET (PIL, same as training/testing)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

# =====================================================
# 4. TRANSFORM (IDENTICAL TO TRAIN/VAL TEST)
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

dataset = UCSDDataset(CSV_PATH)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pil_collate_fn,
)

# =====================================================
# 5. GRADAUG MODEL DEFINITION (EXACT)
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def _forward_stem(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x

    def _head(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def forward_full(self, x):
        x = self._forward_stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self._head(x)

# =====================================================
# 6. LOAD CHECKPOINT
# =====================================================
print(f"\n--- Loading GradAug model ---")
print(CHECKPOINT_PATH)

model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False)
model = model.to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)

model.eval()

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        labels = labels.numpy()

        imgs = torch.stack(
            [test_transform(img) for img in imgs_pil]
        ).to(DEVICE)

        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels)

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 8. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )
specificity = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"GradAug EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



--- Loading GradAug model ---
/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth


Inference: 100%|██████████| 180/180 [00:43<00:00,  4.10it/s]


GradAug EVALUATION — epoch_9.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.7277
AUC                            | 0.8209
F1                             | 0.7864
F2 (β=2)                       | 0.8806
Macro F1                       | 0.7055
Macro F2 (β=2)                 | 0.7297
Precision                      | 0.6674
Recall                         | 0.9570
Macro Precision                | 0.7885
Macro Recall                   | 0.7163
Specificity                    | 0.4756
NPV                            | 0.9096
Cohen Kappa                    | 0.4422
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.349
Singleton (%) (τ=0.9)          | 65.14
ECE                            | 0.1657


In [6]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_onlyvalM_5e4_GradAug_RC_va/epoch_25.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_onlyvalM_5e4_GradAug_RC_va/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 3. METRIC HELPERS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. DATASET & TRANSFORM
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return img, label

def pil_collate_fn(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels, dtype=torch.long)

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=pil_collate_fn)

# =====================================================
# 5. GRADAUG MODEL DEFINITION
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)
        self.conv1, self.bn1, self.relu, self.maxpool = base.conv1, base.bn1, base.relu, base.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = base.layer1, base.layer2, base.layer3, base.layer4
        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def forward_full(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# =====================================================
# 6. LOAD CHECKPOINT & VA
# =====================================================
print(f"\n--- Loading GradAug Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()

print(f"--- Loading Venn-Abers: {os.path.basename(VA_PICKLE_PATH)} ---")
with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 7. INFERENCE
# =====================================================
probs_all, labels_all = [], []
with torch.no_grad():
    for imgs_pil, labels in tqdm(loader, desc="Inference"):
        imgs = torch.stack([test_transform(img) for img in imgs_pil]).to(DEVICE)
        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 8. RESULTS DISPLAY
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"GradAug EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)


--- Loading GradAug Model: epoch_25.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference: 100%|██████████| 180/180 [00:49<00:00,  3.67it/s]



Applying Venn-Abers Calibration...

GradAug EVALUATION: epoch_25.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.5982          | 0.6050         
AUC                            | 0.6630          | 0.6621         
F1                             | 0.6727          | 0.7048         
F2 (β=2)                       | 0.5867          | 0.5764         
Macro F1                       | 0.5762          | 0.5539         
Macro F2 (β=2)                 | 0.5792          | 0.5648         
Precision                      | 0.5865          | 0.5790         
Recall                         | 0.7887          | 0.9007         
Macro Precision                | 0.6062          | 0.6492         
Macro Recall                   | 0.5887          | 0.5903         
Specificity                    | 0.3888          | 0.2800         
NPV                            | 0.6260          |

In [ ]:
# NEH Model with just resize on 96k ucsd samples

# ============================================================
# GradAug EVALUATION — epoch_9.pth
# ============================================================
# Metric                         | Value
# ------------------------------------------------------------
# Accuracy                       | 0.7298
# AUC                            | 0.9165
# F1                             | 0.7929
# F2 (β=2)                       | 0.8965
# Macro F1                       | 0.7020
# Macro F2 (β=2)                 | 0.7326
# Precision                      | 0.6650
# Recall                         | 0.9819
# Macro Precision                | 0.8110
# Macro Recall                   | 0.7153
# Specificity                    | 0.4488
# NPV                            | 0.9570
# Cohen Kappa                    | 0.4428
# ------------------------------------------------------------
# Set Size (avg, τ=0.9)          | 1.267
# Singleton (%) (τ=0.9)          | 73.32
# ECE                            | 0.1869
# ============================================================

In [ ]:
# Exact LR speckleNoise

# ======================================================================
# GRADAUG MODEL — UCSD TEST EVALUATION
# ======================================================================
# Metric                              | Value
# ----------------------------------------------------------------------
# Accuracy                            | 0.7956
# ROC AUC                             | 0.9207
# F1                                  | 0.8239
# Macro F1                            | 0.7901
# Macro Precision                     | 0.8267
# Macro Recall                        | 0.7944
# Specificity                         | 0.6394
# NPV                                 | 0.9257
# Cohen Kappa                         | 0.5902
# ----------------------------------------------------------------------
# Avg Set Size (τ=0.9)                | 1.167
# Singleton Rate % (τ=0.9)            | 83.29
# ECE                                 | 0.1507
# ======================================================================

# only Resize


# ======================================================================
# GRADAUG MODEL — UCSD TEST EVALUATION
# ======================================================================
# Metric                              | Value
# ----------------------------------------------------------------------
# Accuracy                            | 0.8514
# ROC AUC                             | 0.9342
# F1                                  | 0.8658
# Macro F1                            | 0.8496
# Macro Precision                     | 0.8665
# Macro Recall                        | 0.8506
# Specificity                         | 0.7491
# NPV                                 | 0.9391
# Cohen Kappa                         | 0.7022
# ----------------------------------------------------------------------
# Avg Set Size (τ=0.9)                | 1.178
# Singleton Rate % (τ=0.9)            | 82.21
# ECE                                 | 0.0915
# ======================================================================

# only Resize modified

# ======================================================================
# GRADAUG MODEL — UCSD TEST EVALUATION
# ======================================================================
# Metric                              | Value
# ----------------------------------------------------------------------
# Accuracy                            | 0.7267
# ROC AUC                             | 0.9140
# F1                                  | 0.7834
# Macro F1                            | 0.7065
# Macro Precision                     | 0.8065
# Macro Recall                        | 0.7248
# Specificity                         | 0.4682
# NPV                                 | 0.9610
# Cohen Kappa                         | 0.4512
# ----------------------------------------------------------------------
# Avg Set Size (τ=0.9)                | 1.272
# Singleton Rate % (τ=0.9)            | 72.81
# ECE                                 | 0.1878
# ======================================================================

# resize with less speckle 

# ======================================================================
# GRADAUG MODEL — UCSD TEST EVALUATION
# ======================================================================
# Metric                              | Value
# ----------------------------------------------------------------------
# Accuracy                            | 0.8489
# ROC AUC                             | 0.9525
# F1                                  | 0.8662
# Macro F1                            | 0.8463
# Macro Precision                     | 0.8713
# Macro Recall                        | 0.8480
# Specificity                         | 0.7253
# NPV                                 | 0.9606
# Cohen Kappa                         | 0.6972
# ----------------------------------------------------------------------
# Avg Set Size (τ=0.9)                | 1.232
# Singleton Rate % (τ=0.9)            | 76.76
# ECE                                 | 0.0779
# ======================================================================

# resize with less speckle 




--- Loading MC Dropout Model: epoch_9.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference (MC Dropout):   1%|          | 1/90 [00:02<03:18,  2.23s/it]


KeyboardInterrupt: 